# Sound & Loudness in *How It Happened* (Doyle 1913)

Visual exploration of the sound-event annotations (`<ambient_sound>`, `<character_sound>`) and the dictionary-based loudness levels written back into the TEI file by the loudness-level labeling notebook.

The notebook works on **one or several** annotated TEI files. With several files, every figure gets a dropdown to switch between texts, like the interactive `MMiA_sounds_interactive_st_dev.html`.

| # | Figure | Question it answers |
|---|---|---|
| 1 | Loudness distribution per scene (box plots) | Same view as the MMiA HTML: how loud, and how varied, is each scene? |
| 2 | Soundscape timeline | *Where* in the narrative do sounds occur, and how loud are they? |
| 3 | Loudness standard deviation per scene | The *Between Whispers and Screams* proxy: which scenes swing between quiet and loud? |
| 4 | Sound density per scene | How much sound does a scene contain, relative to its length? |
| 5 | Annotated text view | The spans themselves, highlighted in the text by class and loudness |

All figures are exported to `viz_output/` as standalone interactive HTML files (and as PNG if `kaleido` + Chrome are available) for the slides.

## 0 · Setup

In [ ]:
import re, html, math
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# ---- paths ---------------------------------------------------------------
DATA_DIR   = Path('/Users/sguhr/Desktop/for_loudness_level_anno')                        # folder with the loudness-annotated TEI file(s)
XML_FILES  = sorted(DATA_DIR.glob('*.xml'))   # or e.g. [DATA_DIR / 'Doyle_1913_How_it_happened.xml']
CSV_FILE   = DATA_DIR / 'Doyle_1913_How_it_happened_LL_predicted.csv'   # optional cross-check
OUT_DIR    = Path('viz_output'); OUT_DIR.mkdir(exist_ok=True)

# ---- optional human-readable scene labels (per file stem) ------------------
SCENE_LABELS = {
    'Doyle_1913_How_it_happened': {
        0: 'Non-scene',
        1: 'Memory',
        2: 'Station and new car',
        3: 'Brakes fail and crash',
        4: 'After the crash',
    },
}

# ---- look ------------------------------------------------------------------
CLASSES      = ['character_sound', 'ambient_sound']
CLASS_LABEL  = {'character_sound': 'Character sound', 'ambient_sound': 'Ambient sound'}
CLASS_COLOR  = {'character_sound': '#2a78d6', 'ambient_sound': '#eb6834'}   # CVD-validated pair
CLASS_SYMBOL = {'character_sound': 'circle', 'ambient_sound': 'diamond'}   # secondary encoding
INK, INK_2, INK_3 = '#0b0b0b', '#52514e', '#8a8984'
SURFACE, GRID, BAND = '#fcfcfb', '#e9e8e4', 'rgba(82,81,78,0.06)'
FONT = 'Inter, Helvetica Neue, Helvetica, Arial, sans-serif'

pio.templates['talk'] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=15, color=INK),
    title=dict(font=dict(size=20, color=INK), x=0.01, xanchor='left'),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    xaxis=dict(showgrid=False, linecolor=GRID, tickcolor=GRID, ticks='outside',
               tickfont=dict(color=INK_2), title=dict(font=dict(color=INK_2))),
    yaxis=dict(gridcolor=GRID, zeroline=False, linecolor=GRID,
               tickfont=dict(color=INK_2), title=dict(font=dict(color=INK_2))),
    # legend pinned to the bottom edge of the figure, centred -> never cut off, whatever the width
    legend=dict(orientation='h', yref='container', y=0.02, yanchor='bottom', xanchor='center', x=0.5,
                font=dict(color=INK_2, size=14), bgcolor='rgba(0,0,0,0)'),
    hoverlabel=dict(bgcolor='white', bordercolor=GRID, font=dict(family=FONT, size=13, color=INK)),
    margin=dict(l=70, r=30, t=100, b=130),
))
pio.templates.default = 'talk'
print(f'{len(XML_FILES)} TEI file(s):', [p.name for p in XML_FILES])

1 TEI file(s): ['Doyle_1913_How_it_happened.xml']


## 1 · Read the TEI files

The parser walks the document in reading order and records, for every sound event, its scene, class, span text, loudness value, and **word position** in the text (the x-axis of the timeline).

**Repair step:** the file contains duplicated attributes such as `loudness="3.0" loudness="3.0"`. This is not well-formed XML and makes `ElementTree` fail. It comes from running the write-back cell of the labeling notebook twice, because its regex *adds* a `loudness` attribute instead of replacing an existing one. The cell below removes the duplicates in memory; the source file is not touched. See the note at the end for a permanent fix.

In [ ]:
TEI_NS = '{http://www.tei-c.org/ns/1.0}'
SOUND_TAGS = set(CLASSES)
WORD_RE = re.compile(r"[\w'’]+(?:-[\w'’]+)*")

def local(tag):
    return tag.split('}', 1)[-1]

def repair_duplicate_attributes(raw):
    # keep the first occurrence of any attribute that is repeated on the same element
    pattern = re.compile(r'(\s([\w:.-]+)="[^"]*")((?:\s\2="[^"]*")+)')
    return pattern.subn(r'\1', raw)

def load_tei(path):
    raw = Path(path).read_text(encoding='utf-8')
    fixed, n_fixed = repair_duplicate_attributes(raw)
    root = ET.fromstring(fixed)
    body = root.find(f'.//{TEI_NS}body')
    if body is None:
        body = root

    tokens, events, scenes = [], [], []
    state = {'scene': 0}

    def add_text(txt):
        if txt:
            tokens.extend(WORD_RE.findall(txt))

    def walk(el):
        tag = local(el.tag)
        is_scene = tag == 'scene'
        if is_scene:
            sid = el.get('counter') or el.get('n') or len(scenes) + 1
            prev_scene = state['scene']
            state['scene'] = int(sid)
            start = len(tokens)
        if tag in SOUND_TAGS:
            span = re.sub(r'\s+', ' ', ''.join(el.itertext())).strip()
            loud = el.get('loudness')
            events.append(dict(
                scene=state['scene'], sound_class=tag, span=span,
                loudness=float(loud) if loud not in (None, '', 'None') else np.nan,
                word_start=len(tokens), n_words=len(WORD_RE.findall(span)),
            ))
        add_text(el.text)
        for child in el:
            walk(child)
            add_text(child.tail)
        if is_scene:
            scenes.append(dict(scene=state['scene'], word_start=start, word_end=len(tokens)))
            state['scene'] = prev_scene

    walk(body)

    stem = Path(path).stem
    ev = pd.DataFrame(events, columns=['scene', 'sound_class', 'span', 'loudness', 'word_start', 'n_words'])
    sc = pd.DataFrame(scenes).sort_values('scene').reset_index(drop=True)
    total = len(tokens)
    ev['file'] = sc['file'] = stem
    ev['pos_pct'] = (ev['word_start'] + ev['n_words'] / 2) / total * 100
    ev['context'] = [
        ' '.join(tokens[max(0, s - 10): s]) + ' «' + ' '.join(tokens[s: s + n]) + '» ' + ' '.join(tokens[s + n: s + n + 10])
        for s, n in zip(ev['word_start'], ev['n_words'])
    ]
    sc['n_words'] = sc['word_end'] - sc['word_start']
    sc['start_pct'] = sc['word_start'] / total * 100
    sc['end_pct'] = sc['word_end'] / total * 100
    labels = SCENE_LABELS.get(stem, {})
    sc['label'] = [labels.get(s, f'Scene {s}') for s in sc['scene']]
    ev['scene_label'] = ev['scene'].map(dict(zip(sc['scene'], sc['label'])))
    return dict(stem=stem, events=ev, scenes=sc, total_words=total, repaired=n_fixed, root=root)

corpus = [load_tei(p) for p in XML_FILES]
for d in corpus:
    print(f"{d['stem']}: {d['total_words']:,} words · {len(d['scenes'])} scenes · "
          f"{len(d['events'])} sound events · {d['repaired']} duplicated attribute(s) repaired")

Doyle_1913_How_it_happened: 1,404 words · 5 scenes · 26 sound events · 0 duplicated attribute(s) repaired


A quick look at the extracted events:

In [ ]:
events = pd.concat([d['events'] for d in corpus], ignore_index=True)
scenes = pd.concat([d['scenes'] for d in corpus], ignore_index=True)
events[['file', 'scene', 'scene_label', 'sound_class', 'span', 'loudness', 'pos_pct']].round(1)

,file,scene,scene_label,sound_class,span,loudness,pos_pct
0,Doyle_1913_How_it_happened,1,Memory,character_sound,also asking Perkins,3.0,12.2
1,Doyle_1913_How_it_happened,1,Memory,character_sound,and his saying,3.0,12.9
2,Doyle_1913_How_it_happened,1,Memory,character_sound,said I,3.0,13.7
3,Doyle_1913_How_it_happened,1,Memory,character_sound,said he,3.0,14.8
4,Doyle_1913_How_it_happened,1,Memory,character_sound,said I,3.0,15.9
5,Doyle_1913_How_it_happened,2,Station and new car,ambient_sound,so I clapped on both brakes,3.5,30.5
6,Doyle_1913_How_it_happened,2,Station and new car,ambient_sound,and the lever clanged to its full limit withou...,4.0,33.2
7,Doyle_1913_How_it_happened,2,Station and new car,character_sound,said he,3.0,42.8
8,Doyle_1913_How_it_happened,2,Station and new car,character_sound,said he,3.0,47.1
9,Doyle_1913_How_it_happened,2,Station and new car,character_sound,said I,3.0,48.6


### Cross-check against the extraction CSV (optional)

Makes sure the parser finds the same spans as the labeling notebook.

In [ ]:
import ast
if CSV_FILE.exists():
    csv = pd.read_csv(CSV_FILE)
    for _, row in csv.iterrows():
        stem = Path(row['File']).stem
        ev = events[events['file'] == stem]
        for col, cls in [('Ambient Sound Spans', 'ambient_sound'), ('Character Sound Spans', 'character_sound')]:
            csv_spans = ast.literal_eval(row[col])
            xml_spans = ev.loc[ev['sound_class'] == cls, 'span'].tolist()
            status = 'match' if csv_spans == xml_spans else 'DIFFERENT'
            print(f'{stem} · {cls:16s} CSV {len(csv_spans):3d} | XML {len(xml_spans):3d} → {status}')
else:
    print('No CSV found — skipped.')

Doyle_1913_How_it_happened · ambient_sound    CSV   6 | XML   7 → DIFFERENT
Doyle_1913_How_it_happened · character_sound  CSV  20 | XML  19 → DIFFERENT


### Events without a loudness value

These spans matched no key of the loudness dictionary. They count toward sound **density** (Fig. 4) but are excluded from all loudness statistics. In *How It Happened* this is only `so I clapped on both brakes`, a real sound event that the dictionary misses (*clap* / *clapped* is not a key). Adding `'clap': 4` to the dictionary would label it.

In [ ]:
events[events['loudness'].isna()][['file', 'scene', 'sound_class', 'span', 'context']]

,file,scene,sound_class,span,context


### Scene summary table

In [ ]:
def scene_summary(ev, sc):
    rows = []
    for _, s in sc.iterrows():
        e = ev[(ev['file'] == s['file']) & (ev['scene'] == s['scene'])]
        v = e['loudness'].dropna()
        rows.append({
            'file': s['file'], 'scene': s['scene'], 'label': s['label'], 'words': s['n_words'],
            'events': len(e),
            'character': (e['sound_class'] == 'character_sound').sum(),
            'ambient': (e['sound_class'] == 'ambient_sound').sum(),
            'events / 100 words': round(len(e) / s['n_words'] * 100, 2) if s['n_words'] else np.nan,
            'mean loudness': round(v.mean(), 2) if len(v) else np.nan,
            'max loudness': v.max() if len(v) else np.nan,
            'std (all)': round(v.std(ddof=0), 2) if len(v) > 1 else np.nan,
            'std (ambient)': round(e.loc[e.sound_class == 'ambient_sound', 'loudness'].dropna().std(ddof=0), 2)
                             if e.loc[e.sound_class == 'ambient_sound', 'loudness'].notna().sum() > 1 else np.nan,
            'std (character)': round(e.loc[e.sound_class == 'character_sound', 'loudness'].dropna().std(ddof=0), 2)
                             if e.loc[e.sound_class == 'character_sound', 'loudness'].notna().sum() > 1 else np.nan,
        })
    return pd.DataFrame(rows)

summary = scene_summary(events, scenes)
summary

,file,scene,label,words,events,character,ambient,events / 100 words,mean loudness,max loudness,std (all),std (ambient),std (character)
0,Doyle_1913_How_it_happened,0,Non-scene,86,0,0,0,0.00,NaN,NaN,NaN,NaN,NaN
1,Doyle_1913_How_it_happened,1,Memory,135,5,5,0,3.70,3.00,3.0,0.00,NaN,0.00
2,Doyle_1913_How_it_happened,2,Station and new car,598,7,4,3,1.17,3.36,4.0,0.44,0.24,0.00
3,Doyle_1913_How_it_happened,3,Brakes fail and crash,177,1,0,1,0.56,5.00,5.0,NaN,NaN,NaN
4,Doyle_1913_How_it_happened,4,After the crash,395,13,10,3,3.29,3.08,4.0,0.38,0.62,0.27


## Helpers: one figure, several files

Each figure is built by a function that takes one file's data and returns traces, shapes and annotations. `multi_file_figure` stacks them and adds a dropdown when more than one file is loaded, the same mechanism as in the MMiA HTML.

In [ ]:
def _plain(v):
    # plotly >= 6 sends numpy/pandas data as base64 'bdata'; older notebook front-ends
    # cannot decode it and silently draw empty traces -> rebuild every trace from plain lists
    if isinstance(v, dict):
        return {k: _plain(x) for k, x in v.items()}
    if isinstance(v, (pd.Series, pd.Index)):
        v = v.to_numpy()
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    if isinstance(v, np.generic):
        return v.item()
    return v

def plain_trace(t):
    return type(t)(**_plain(t.to_plotly_json()))

def multi_file_figure(builder, title, **layout):
    fig = go.Figure()
    per_file = []
    for i, d in enumerate(corpus):
        traces, shapes, annotations = builder(d)
        first = len(fig.data)
        for t in map(plain_trace, traces):
            t.visible = (i == 0)
            fig.add_trace(t)
        per_file.append((d['stem'], range(first, len(fig.data)), shapes, annotations))

    n = len(fig.data)
    stem0, _, shapes0, ann0 = per_file[0]
    fig.update_layout(title=f'{title}<br><sup style="color:{INK_2}">{stem0}</sup>',
                      shapes=shapes0, annotations=ann0, **layout)
    if len(per_file) > 1:
        buttons = []
        for stem, idx, shapes, ann in per_file:
            vis = [j in idx for j in range(n)]
            buttons.append(dict(label=stem, method='update', args=[
                {'visible': vis},
                {'title': f'{title}<br><sup style="color:{INK_2}">{stem}</sup>', 'shapes': shapes, 'annotations': ann},
            ]))
        fig.update_layout(updatemenus=[dict(buttons=buttons, direction='down', x=1, xanchor='right',
                                            y=1.22, yanchor='top', bgcolor='white', bordercolor=GRID)])
    return fig

def export(fig, name, width=1400, height=650):
    fig.write_html(OUT_DIR / f'{name}.html', include_plotlyjs=True)
    try:
        fig.write_image(OUT_DIR / f'{name}.png', width=width, height=height, scale=2)
        print(f'saved {name}.html + .png')
    except Exception as e:
        print(f'saved {name}.html (PNG skipped: {type(e).__name__}; use the camera icon in the plot toolbar)')

def wrap(s, width=60):
    words, lines, cur = s.split(), [], ''
    for w in words:
        if len(cur) + len(w) + 1 > width:
            lines.append(cur); cur = w
        else:
            cur = f'{cur} {w}'.strip()
    lines.append(cur)
    return '<br>'.join(lines)

## Fig. 1 · Loudness distribution per scene

Equivalent to the MMiA visualization: one box per scene and sound class, with every sound event shown as a point (hover shows the span). Scenes without sound events stay on the axis, so silence remains visible.

In [ ]:
def build_boxes(d):
    ev, sc = d['events'].dropna(subset=['loudness']), d['scenes']
    order = [str(s) for s in sc['scene']]
    traces = []
    for cls in CLASSES:
        e = ev[ev['sound_class'] == cls]
        traces.append(go.Box(
            x=e['scene'].astype(str), y=e['loudness'], name=CLASS_LABEL[cls], legendgroup=cls,
            boxpoints='all', jitter=0.5, pointpos=0,
            marker=dict(color=CLASS_COLOR[cls], size=9, symbol=CLASS_SYMBOL[cls],
                        line=dict(color=SURFACE, width=1.5)),
            line=dict(color=CLASS_COLOR[cls], width=2), fillcolor='rgba(0,0,0,0)',
            text=[wrap(s) for s in e['span']],
            customdata=e[['scene_label']].values,
            hovertemplate='<b>%{text}</b><br>Scene %{x} · %{customdata[0]}<br>Loudness %{y}<extra>' + CLASS_LABEL[cls] + '</extra>',
        ))
    # category axis incl. silent scenes, labels "1<br>Frame: memory"
    ticktext = [f'<b>{s}</b><br><span style="font-size:12px">{l}</span>' for s, l in zip(sc['scene'], sc['label'])]
    d['_box_axis'] = dict(categoryarray=order, ticktext=ticktext, tickvals=order)
    # invisible anchor so that scenes without sound keep their slot on the axis
    traces.append(go.Scatter(x=order, y=[-10] * len(order), mode='markers', showlegend=False,
                             hoverinfo='skip', marker=dict(opacity=0)))
    # note: on a category axis, annotation x is the category *index*
    ann = [dict(x=i, y=0.4, xref='x', yref='y', showarrow=False, text='no sound events',
                font=dict(size=13, color=INK_3))
           for i, s in enumerate(sc['scene']) if (d['events']['scene'] == s).sum() == 0]
    return traces, [], ann

fig1 = multi_file_figure(build_boxes, 'Loudness distribution per scene', boxmode='group', boxgap=0.35, boxgroupgap=0.15,
                         height=620, yaxis=dict(title='Loudness level', range=[-0.3, 5.5], dtick=1))
ax = corpus[0]['_box_axis']
fig1.update_xaxes(type='category', categoryorder='array', title='Scene', **ax)
fig1.show()
export(fig1, 'fig1_loudness_per_scene_box')

saved fig1_loudness_per_scene_box.html + .png


## Fig. 2 · Soundscape timeline

Every sound event at its position in the text (x = % of the story's words), with scenes as bands. For *How It Happened* this shows the arc of the story in sound. The non-scene opening is silent. Speech tags stay at a steady 3. The ambient sounds climb as the brakes fail (the clanging lever and the creaking body, both 4) and peak at *I heard the crash* (5). After the crash come hushed voices (2.5) and cries (3.5–4) around a narrator whom nobody hears (*they did not seem to hear me*).

Events without a loudness value are not shown here; they are listed in section 1 and counted in Fig. 4.

In [ ]:
def build_timeline(d):
    ev, sc = d['events'], d['scenes']
    traces, shapes, ann = [], [], []
    for cls in CLASSES:
        e = ev[(ev['sound_class'] == cls) & ev['loudness'].notna()]
        traces.append(go.Scatter(
            x=e['pos_pct'], y=e['loudness'], mode='markers', cliponaxis=False, name=CLASS_LABEL[cls], legendgroup=cls,
            marker=dict(color=CLASS_COLOR[cls], size=13, symbol=CLASS_SYMBOL[cls],
                        line=dict(color=SURFACE, width=2)),
            text=[wrap(s) for s in e['span']], customdata=np.stack([e['scene_label'], [wrap(c, 70) for c in e['context']]], axis=-1),
            hovertemplate='<b>%{text}</b><br>Loudness %{y} · %{customdata[0]}<br><span style="color:#52514e">%{customdata[1]}</span><extra></extra>',
        ))
    for i, s in sc.iterrows():
        if i % 2 == 0:
            shapes.append(dict(type='rect', xref='x', yref='paper', x0=s['start_pct'], x1=s['end_pct'], y0=0, y1=1,
                               fillcolor=BAND, line_width=0, layer='below'))
        ann.append(dict(x=(s['start_pct'] + s['end_pct']) / 2, y=1.0, xref='x', yref='paper', yanchor='bottom',
                        text=f"<b>{s['scene']}</b> {s['label']}", showarrow=False,
                        font=dict(size=12, color=INK_2)))
    # annotate the loudest event
    v = ev.dropna(subset=['loudness'])
    if len(v):
        top = v.loc[v['loudness'].idxmax()]
        ann.append(dict(x=top['pos_pct'], y=top['loudness'], text=f"“{top['span']}”", showarrow=True,
                        arrowhead=0, arrowcolor=INK_3, ax=-80, ay=-10, font=dict(size=13, color=INK),
                        xanchor='right'))
    return traces, shapes, ann

fig2 = multi_file_figure(build_timeline, 'Soundscape timeline', height=560,
                         xaxis=dict(title='Position in the narrative (% of words)', range=[0, 100], ticksuffix='%'),
                         yaxis=dict(title='Loudness level', range=[-0.2, 5.6], dtick=1),
                         margin=dict(t=120))
fig2.update_layout(
    legend=dict(orientation='h', x=1, xanchor='right', y=1.08, yanchor='bottom'),
    margin=dict(t=150, r=40),
)
fig2.show()
export(fig2, 'fig2_soundscape_timeline', height=560)


saved fig2_soundscape_timeline.html + .png


## Fig. 3 · Loudness standard deviation per scene

The proxy from *Between Whispers and Screams* (NLP4DH 2026): a high standard deviation of loudness marks scenes that swing between quiet and loud. Hover shows *n*. The std is only drawn where a scene has at least two valued events of that class; scenes with fewer than two valued events are marked "no std".

In [ ]:
def build_std(d):
    ev, sc = d['events'].dropna(subset=['loudness']), d['scenes']
    traces, ann = [], []
    x = [str(s) for s in sc['scene']]
    series = [('all', 'All sounds', INK_3)] + [(c, CLASS_LABEL[c], CLASS_COLOR[c]) for c in CLASSES]
    for key, label, color in series:
        ys, ns = [], []
        for s in sc['scene']:
            e = ev[ev['scene'] == s] if key == 'all' else ev[(ev['scene'] == s) & (ev['sound_class'] == key)]
            ns.append(len(e))
            ys.append(e['loudness'].std(ddof=0) if len(e) > 1 else None)
        traces.append(go.Bar(
            x=x, y=ys, name=label, marker=dict(color=color, line=dict(color=SURFACE, width=2), cornerradius=4),
            customdata=np.array(ns), text=[f'{y:.2f}' if y is not None else '' for y in ys], textposition='outside',
            textfont=dict(color=INK_2, size=12), cliponaxis=False,
            hovertemplate=label + ' · scene %{x}<br>std %{y:.2f} (n = %{customdata})<extra></extra>',
        ))
    ticktext = [f'<b>{s}</b><br><span style="font-size:12px">{l}</span>' for s, l in zip(sc['scene'], sc['label'])]
    d['_std_axis'] = dict(tickvals=x, ticktext=ticktext)
    for i, s in enumerate(sc['scene']):
        n = (ev['scene'] == s).sum()
        if n < 2:
            ann.append(dict(x=i, y=0, xref='x', yref='y', yanchor='bottom', showarrow=False,
                            text=f'n = {n}<br>(no std)', font=dict(size=13, color=INK_3)))
    return traces, [], ann

fig3 = multi_file_figure(build_std, 'Loudness standard deviation per scene', barmode='group', bargap=0.3, bargroupgap=0.08,
                         height=560, yaxis=dict(title='Std of loudness level', rangemode='tozero'))
fig3.update_xaxes(type='category', title='Scene', **corpus[0]['_std_axis'])
fig3.update_layout(
    legend=dict(orientation='h', x=1, xanchor='right', y=1.08, yanchor='bottom'),
    margin=dict(t=150, r=40))
fig3.show()
export(fig3, 'fig3_loudness_std_per_scene', height=560)

saved fig3_loudness_std_per_scene.html + .png


## Fig. 4 · Sound density per scene

Sound events per 100 words, split by class, so that long and short scenes can be compared. Includes events without a loudness value.

In [ ]:
def build_density(d):
    ev, sc = d['events'], d['scenes']
    x = [str(s) for s in sc['scene']]
    traces = []
    for cls in CLASSES:
        counts = [((ev['scene'] == s) & (ev['sound_class'] == cls)).sum() for s in sc['scene']]
        dens = [c / w * 100 if w else 0 for c, w in zip(counts, sc['n_words'])]
        traces.append(go.Bar(
            x=x, y=dens, name=CLASS_LABEL[cls], marker=dict(color=CLASS_COLOR[cls], line=dict(color=SURFACE, width=2)),
            customdata=np.stack([counts, sc['n_words']], axis=-1),
            hovertemplate=CLASS_LABEL[cls] + ' · scene %{x}<br>%{y:.2f} per 100 words<br>%{customdata[0]} events in %{customdata[1]} words<extra></extra>',
        ))
    ticktext = [f'<b>{s}</b><br><span style="font-size:12px">{l}</span>' for s, l in zip(sc['scene'], sc['label'])]
    d['_dens_axis'] = dict(tickvals=x, ticktext=ticktext)
    ann = [dict(x=i, y=0, xref='x', yref='y', yanchor='bottom', showarrow=False, text='no_sound',
                font=dict(size=13, color=INK_3)) for i, s in enumerate(sc['scene']) if (ev['scene'] == s).sum() == 0]
    return traces, [], ann

fig4 = multi_file_figure(build_density, 'Sound density per scene', barmode='stack', bargap=0.35, height=520, legend=dict(traceorder='normal'),
                         yaxis=dict(title='Sound events per 100 words', rangemode='tozero'))
fig4.update_xaxes(type='category', title='Scene', **corpus[0]['_dens_axis'])
fig4.update_layout(
    legend=dict(orientation='h', x=1, xanchor='right', y=1.08, yanchor='bottom'),
    margin=dict(t=150, r=40))
fig4.show()
export(fig4, 'fig4_sound_density_per_scene', height=520)

saved fig4_sound_density_per_scene.html + .png


## Fig. 5 · Annotated text view

The story with its sound events highlighted: color = class, saturation = loudness (paler = quieter, grey dashed underline = no value). Hover a span to see its loudness. Useful as a close-reading companion slide or screenshot. The same view is saved to `viz_output/fig5_annotated_text_<file>.html`.

In [ ]:
def rgba(hex_, a):
    h = hex_.lstrip('#')
    return f'rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{a:.2f})'

def render_annotated(d):
    labels = dict(zip(d['scenes']['scene'], d['scenes']['label']))
    def esc(t):
        return html.escape(t or '').replace('\n\n', '</p><p>').replace('\n', ' ')
    def walk(el):
        tag = local(el.tag)
        inner = esc(el.text) + ''.join(walk(c) + esc(c.tail) for c in el)
        if tag in SOUND_TAGS:
            loud = el.get('loudness')
            col = CLASS_COLOR[tag]
            if loud:
                a = 0.12 + 0.13 * float(loud)          # 0 → .12, 5 → .77
                style = f'background:{rgba(col, a)};border-bottom:2px solid {col};'
                tip = f'{CLASS_LABEL[tag]} · loudness {loud}'
                badge = f'<sup style="color:{col};font-weight:700;font-size:10px;margin-left:2px">{float(loud):g}</sup>'
            else:
                style = f'border-bottom:2px dashed {INK_3};'
                tip = f'{CLASS_LABEL[tag]} · no loudness value'
                badge = ''
            return f'<span class="snd" style="{style}" title="{tip}">{inner}</span>{badge}'
        if tag == 'scene':
            sid = int(el.get('counter') or el.get('n') or 0)
            return (f'<section><div class="sh">Scene {sid} · {html.escape(labels.get(sid, ""))}</div>'
                    f'<p>{inner}</p></section>')
        return inner
    body = d['root'].find(f'.//{TEI_NS}body')
    body = d['root'] if body is None else body
    legend = ''.join(
        f'<span class="snd" style="background:{rgba(CLASS_COLOR[c], .45)};border-bottom:2px solid {CLASS_COLOR[c]}">{CLASS_LABEL[c]}</span> '
        for c in CLASSES) + f'<span class="snd" style="border-bottom:2px dashed {INK_3}">no loudness value</span>'
    return f'''
<div class="annot">
<style>
 .annot {{font-family: Georgia, 'Times New Roman', serif; font-size:16px; line-height:1.75; color:{INK};
          background:{SURFACE}; max-width:820px; padding:24px 32px; border:1px solid {GRID}; border-radius:8px}}
 .annot .sh {{font-family:{FONT}; font-size:12px; letter-spacing:.06em; text-transform:uppercase; color:{INK_2};
              margin:22px 0 4px; padding-top:10px; border-top:1px solid {GRID}}}
 .annot .snd {{padding:1px 3px; border-radius:3px}}
 .annot .lg {{font-family:{FONT}; font-size:13px; color:{INK_2}; margin-bottom:8px}}
 .annot p {{margin:0 0 10px}}
</style>
<div class="lg">{legend}</div>
<p>{walk(body)}</p>
</div>'''

for d in corpus:
    page = render_annotated(d)
    (OUT_DIR / f"fig5_annotated_text_{d['stem']}.html").write_text(
        f'<!doctype html><meta charset="utf-8"><title>{d["stem"]}</title><body style="background:{SURFACE}">{page}</body>',
        encoding='utf-8')
    display(HTML(page))

## Talking points for *How It Happened*

Computed from the data above. Check the numbers after you correct any annotations.

In [ ]:
for d in corpus:
    ev, s = d['events'], summary[summary['file'] == d['stem']]
    v = ev.dropna(subset=['loudness'])
    print(f"── {d['stem']}")
    print(f"   {len(ev)} sound events ({(ev.sound_class=='character_sound').sum()} character, "
          f"{(ev.sound_class=='ambient_sound').sum()} ambient); {ev['loudness'].isna().sum()} without loudness value")
    silent = s.loc[s['events'] == 0, 'label'].tolist()
    if silent: print(f"   silent scene(s): {', '.join(silent)}")
    top = v.loc[v['loudness'].idxmax()]
    print(f"   loudest event: “{top['span']}” ({top['loudness']:g}) in scene {top['scene']} at {top['pos_pct']:.0f}% of the text")
    for cls in CLASSES:
        c = v[v.sound_class == cls]['loudness']
        print(f"   {CLASS_LABEL[cls]:16s} mean {c.mean():.2f} · std {c.std(ddof=0):.2f} · values {sorted(float(x) for x in c.unique())}")
    dens = s.loc[s['events / 100 words'].idxmax()]
    print(f"   densest scene: {dens['label']} ({dens['events / 100 words']} events / 100 words)")
    print()

── Doyle_1913_How_it_happened
   26 sound events (19 character, 7 ambient); 0 without loudness value
   silent scene(s): Non-scene
   loudest event: “I heard the crash” (5) in scene 3 at 71% of the text
   Character sound  mean 3.03 · std 0.20 · values [2.5, 3.0, 3.5]
   Ambient sound    mean 3.71 · std 0.75 · values [2.5, 3.0, 3.5, 4.0, 5.0]
   densest scene: Memory (3.7 events / 100 words)


## Note: permanent fix for the duplicated `loudness` attributes

In the labeling notebook, the write-back regex appends `loudness="…"` to whatever attributes are already there, so every re-run adds another copy. Replacing the pattern so that an existing `loudness` attribute is dropped first keeps the file well-formed:

```python
pattern = fr'(<(ambient|character)_sound)((?:\s+(?!loudness=)[\w:.-]+="[^"]*")*)(?:\s+loudness="[^"]*")*\s*>(\s*{re.escape(sound_span)}\s*<)'
replacement = fr'\1\3 loudness="{loudness_value}">\4'
```

This version keeps any other attributes, removes every old `loudness` value, and writes exactly one new one.